In [ ]:
import uproot
import pandas as pd
import numpy as np

MAIN_CAF = "/storage/1/st15719/caf_new_sum.2.6M_weighted.root"
PFP_FILE = "reco_pfp_info.root"

Enu_e_had = "rec/common/common.ixn.pandora/common.ixn.pandora.Enu.e_had"
Enu_e_calo = "rec/common/common.ixn.pandora/common.ixn.pandora.Enu.e_calo"
Enu_mu_had = "rec/common/common.ixn.pandora/common.ixn.pandora.Enu.mu_had"
Enu_mu_range = "rec/common/common.ixn.pandora/common.ixn.pandora.Enu.mu_range"
PDG = "rec/mc/mc.nu/mc.nu.pdg"

caf_tree = uproot.open(MAIN_CAF)["cafTree"]
data = caf_tree.arrays([Enu_e_had, Enu_e_calo, Enu_mu_had, Enu_mu_range, PDG])

initial_count = len(data[PDG])
print(f"Initial entries in CAF file: {initial_count}")

df_kinematics = pd.DataFrame({
    'Enu_e_had': [x[0] if len(x) > 0 else np.nan for x in data[Enu_e_had]],
    'Enu_e_calo': [x[0] if len(x) > 0 else np.nan for x in data[Enu_e_calo]],
    'Enu_mu_had': [x[0] if len(x) > 0 else np.nan for x in data[Enu_mu_had]],
    'Enu_mu_range': [x[0] if len(x) > 0 else np.nan for x in data[Enu_mu_range]],
})

df_kinematics['pdg'] = data[PDG]
df_kinematics['inelasticity_e'] = 1 -( (df_kinematics['Enu_e_had'] / df_kinematics['Enu_e_calo']) )
df_kinematics['inelasticity_mu'] = 1 -( (df_kinematics['Enu_mu_had'] / df_kinematics['Enu_mu_range']) )
df_kinematics['label'] = (df_kinematics['pdg'] < 0).astype(int)

pfp_tree = uproot.open(PFP_FILE)["reco_pfp_tree"]
df_pfps = pfp_tree.arrays(library="pd")

df_final = pd.concat([df_kinematics, df_pfps], axis=1)

count_pre_filter = len(df_final)
df_final = df_final[df_final['reco_ok'] == True]
count_after_reco_check = len(df_final)
print(f"Entries discarded by reco_ok filter: {count_pre_filter - count_after_reco_check}")


In [ ]:
df_final = df_final.drop(columns= ['reco_ok','Enu_e_had', 'Enu_e_calo', 'Enu_mu_had', 'Enu_mu_range'])
df_final

In [ ]:
import matplotlib.pyplot as plt

plt.plot(df_final['inelasticity_e'], df_final['pdg'], 'o', alpha=0.5)

df_final